# Worm Moebius

In this notebook, we implement the worm algorithm for the Moebius code for $d = 2 p$ with $p$ odd prime. 

In [1]:
import numpy as np
from coherentinfo.moebius_two_odd_prime import MoebiusCodeTwoOddPrime
from coherentinfo.errormodel import ErrorModelLindbladTwoOddPrime
from coherentinfo.worm import (
    all_move_probabilities,
    single_move_probability,
    stabilizer_edges,
    stab_labels,
    move_error
)
import jax
import jax.numpy as jnp
import numpy as np
from typing import Tuple
from jax import Array
from jax.typing import ArrayLike
from functools import partial

In [42]:
length = 11  # 5
width = 11  # 5
p = 3
d = 2 * p
moebius_code = MoebiusCodeTwoOddPrime(length=length, width=width, d=d)
h_z = moebius_code.h_z
h_x = moebius_code.h_x
h_z_mod_2 = moebius_code.h_z_mod_2
h_z_mod_p = moebius_code.h_z_mod_p
h_x_mod_2 = moebius_code.h_x_mod_2
h_x_mod_p = moebius_code.h_x_mod_p
logical_x = moebius_code.logical_x
logical_z = moebius_code.logical_z
gamma_t = 0.3
em_lindblad = ErrorModelLindbladTwoOddPrime(
    moebius_code.num_edges, d=d, gamma_t=gamma_t
)

base_key = jax.random.PRNGKey(42)
initial_error = em_lindblad.generate_random_error(base_key)
initial_error_mod_2 = jnp.mod(initial_error, 2)
initial_error_mod_p = jnp.mod(initial_error, p)
# Here we consider the full syndrome including the plaquette
# we usually remove because of the constraint as this simplified the
# coding of the worm algorithm. In fact, in this the syndromes will
# always be annihilated in pairs, and the total number of syndromes is
# always even as one can check numerically.
syndrome = moebius_code.get_plaquette_syndrome(initial_error)
syndrome_mod_2 = jnp.mod(syndrome, 2)
syndrome_mod_p = jnp.mod(syndrome, p)
num_plaquette, num_edges = h_x.shape

In [48]:
def random_edge_boundary(key):
    return jax.random.randint(key, 1, 0, 3)


def random_edge_bulk(key):
    return jax.random.randint(key, 1, 0, 4)


# remember to add x when you run with scan
def worm_step(worm_state, x, error_model: ErrorModelLindbladTwoOddPrime):
    # def worm_step(worm_state, error_model: ErrorModelLindbladTwoOddPrime):
    # Some quantities might be fixed but it is just convenient
    # to dump everything in a dictionary

    def do_not_attempt_step(worm_state):
        return worm_state

    def attempt_step(worm_state):
        worm_error = worm_state["worm_error"]
        head = worm_state["head"]
        tail = worm_state["tail"]
        # The stabilizers of the same kind as the error
        h_error_mod_2 = worm_state["h_error_mod_2"]
        h_error_mod_p = worm_state["h_error_mod_p"]
        # The stabilizers that can be changed by the error
        h_mod_2 = worm_state["h_mod_2"]
        h_mod_p = worm_state["h_mod_p"]
        p = worm_state["p"]
        key = worm_state["key"]

        key, subkey = jax.random.split(key)

        head_edges = stabilizer_edges(head, h_x_mod_p)
        random_integer = jax.lax.cond(
            head_edges[-1] == -1,
            random_edge_boundary,
            random_edge_bulk,
            subkey
        )
        edge = head_edges[random_integer][0]
        key, subkey = jax.random.split(key)
        power = jax.random.randint(subkey, 1, 0, p)[0]
        key, subkey = jax.random.split(key)
        stab_bool = jax.random.randint(subkey, 1, 0, 2)[0] == True
        key, subkey = jax.random.split(key)

        # We add these two keys for later convenience
        worm_state["edge"] = edge
        worm_state["power"] = power
        worm_state["stab_bool"] = stab_bool

        prob_move, prob_move_worm_error = single_move_probability(
            edge,
            power,
            worm_error,
            stab_bool,
            h_error_mod_p,
            p,
            error_model
        )

        acceptance_prob = jnp.min(
            jnp.array([1.0, prob_move / prob_move_worm_error]))
        # This is to handle the case in which we have 3 edges which is marked with
        # a -1, and so should always be rejected.
        acceptance_prob = jax.lax.cond(
            prob_move < 0, lambda: 0.0, lambda: acceptance_prob)

        def reject(state):
            state["attempted_moves"] += 1
            return state

        def accept(state):
            head = state["head"]
            p = state["p"]
            h_error_mod_2 = state["h_error_mod_2"]
            h_error_mod_p = state["h_error_mod_p"]
            h_mod_2 = state["h_mod_2"]
            h_mod_p = state["h_mod_p"]
            edge = state["edge"]
            power = state["power"]
            stab_bool = state["stab_bool"]
            worm_error = state["worm_error"]
            head = state["head"]
            # This needs to be taken into account obviously so we should not update
            # if continue_worm is fals
            continue_worm = state["continue_worm"]
            proposed_move = move_error(
                edge, power, stab_bool, h_error_mod_p, p)
            new_worm_error_mod_2 = jnp.mod(
                proposed_move[0, :] + worm_error[0, :], 2)
            new_worm_error_mod_p = jnp.mod(
                proposed_move[1, :] + worm_error[1, :], p)
            new_worm_error = jnp.vstack(
                (new_worm_error_mod_2, new_worm_error_mod_p))
            # We now update the new parameters
            new_state = {}
            new_state["worm_error"] = new_worm_error
            incident_stab = stab_labels(edge, h_mod_p)
            # jax.debug.print("jax.debug.print(x) -> {x}", x=incident_stab)
            new_head = jax.lax.cond(
                incident_stab[0] == head, lambda x: x[1], lambda x: x[0], incident_stab)

            new_state["head"] = new_head
            new_state["tail"] = state["tail"]
            new_state["continue_worm"] = jnp.logical_or(
                new_head != tail, state["accepted_moves"] == 0)
            new_state["h_error_mod_2"] = state["h_error_mod_2"]
            new_state["h_error_mod_p"] = state["h_error_mod_p"]
            new_state["h_mod_2"] = state["h_mod_2"]
            new_state["h_mod_p"] = state["h_mod_p"]
            new_state["p"] = p
            new_state["accepted_moves"] = state["accepted_moves"] + 1
            new_state["attempted_moves"] = state["attempted_moves"] + 1
            new_state["key"] = state["key"]
            new_state["edge"] = edge
            new_state["power"] = power
            new_state["stab_bool"] = stab_bool
            return new_state

        acceptance_random_number = jax.random.uniform(subkey)
        # print("Acceptance random number: {}".format(acceptance_random_number))
        # print("Acceptance probability: {}".format(acceptance_prob))
        accept_condition = acceptance_random_number <= acceptance_prob
        # jax.debug.print("jax.debug.print(x) -> {x}", x=acceptance_prob)

        accept_condition = accept_condition

        # Important to update the key!
        worm_state["key"] = key
        new_worm_state = \
            jax.lax.cond(
                accept_condition,
                accept,
                reject,
                worm_state
            )

        new_worm_state

        new_worm_state.pop("edge", None)
        new_worm_state.pop("power", None)
        new_worm_state.pop("stab_bool", None)
        return new_worm_state

    new_worm_state = jax.lax.cond(
        worm_state["continue_worm"],
        attempt_step,
        do_not_attempt_step,
        worm_state,
    )

    return new_worm_state, None


worm_step_partial = partial(worm_step, error_model=em_lindblad)
max_steps = 10000
initial_worm_state = {}
initial_worm_state["worm_error"] = jnp.vstack(
    (initial_error_mod_2, initial_error_mod_p))
initial_head = 18
initial_worm_state["head"] = initial_head
initial_worm_state["tail"] = initial_head
initial_worm_state["continue_worm"] = True
initial_worm_state["h_error_mod_2"] = h_z_mod_2
initial_worm_state["h_error_mod_p"] = h_z_mod_p
initial_worm_state["h_mod_2"] = h_x_mod_2
initial_worm_state["h_mod_p"] = h_x_mod_p
initial_worm_state["p"] = p
initial_worm_state["accepted_moves"] = 0
initial_worm_state["attempted_moves"] = 0
initial_worm_state["key"] = base_key

new_worm_state = initial_worm_state.copy()

jit_worm_step_partial = jax.jit(worm_step_partial)

# for _ in range(max_steps):
#     new_worm_state, _ = jit_worm_step_partial(new_worm_state)

new_worm_state, _ = jax.lax.scan(
    jit_worm_step_partial, initial_worm_state, jnp.arange(max_steps))

print("Number of accepted moves: {}".format(new_worm_state["accepted_moves"]))
print("Number of attempted moves: {}".format(
    new_worm_state["attempted_moves"]))

Number of accepted moves: 20
Number of attempted moves: 457


In [49]:
print(new_worm_state["head"])
print(initial_worm_state["head"])

18
18


In [50]:
print(jnp.max(
    jnp.abs(jnp.mod(new_worm_state["worm_error"][0, :] - initial_worm_state["worm_error"][0, :], 2))))

1


In [51]:
print(jnp.max(
    jnp.abs(jnp.mod(new_worm_state["worm_error"][1, :] - initial_worm_state["worm_error"][1, :], p))))

2


In [52]:
new_syndrome_mod_2 = jnp.mod(h_x_mod_2 @ new_worm_state["worm_error"][0, :], 2)

jnp.mod(new_syndrome_mod_2 - syndrome_mod_2, 2)

Array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int32)

In [53]:
new_syndrome_mod_p = jnp.mod(h_x_mod_p @ new_worm_state["worm_error"][1, :], p)

jnp.mod(new_syndrome_mod_p - syndrome_mod_p, p)

Array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int32)

In [ ]:
def run_worm(
    base_key: Array,
    initial_error,
    h_x_mod_2: Array,
    h_x_mod_p: Array,
    num_plaquette: int,
    num_edges: int,
    error_model: ErrorModelLindbladTwoOddPrime,
    max_steps: int,
) -> Array:

    def random_edge_boundary(key):
        return jax.random.randint(key, 1, 0, 3)

    def random_edge_bulk(key):
        return jax.random.randint(key, 1, 0, 4)

    def initialize_worm(key):
        """In the first iteration I will initialize a fixed error"""
        pass

    def worm_step(carry_worm_step, x):
        (worm_error_mod_2, worm_error_mod_p, worm_syndrome_mod_2,
            worm_syndrome_mod_p, head, tail, continue_worm, key) = carry_worm_step
        # Split the key into a new key that will get passed at
        # the end and a subkey that will be used in this step
        key, subkey = jax.random.split(key)

        def reject(args):
            pass

        def accept(args):
            pass

Let us break the problem into sub-problems. Let us define a function that given an edge, returns the 
plaquette stabilizers $\mod p$ associated with the two plaquettes that touch it.

Now we want a function that given, a plaquette $\mod p$ returns the probabilities of having an error $\mod 2$ at one of its edges, and all the powers $0, 1, ..., p - 1$ of the plaquette itself. I note that I believe that it is important to choose the moves weighted by these probabilities. In particular, even the random choice of head move, should be weighted, because the plaquette of weight $3$ at the boundary would have a different probability of those of weight $4$ in the bulk. 

While the previous might be useful I think that what we really want is given a head, i.e., a plaquette index to obtain the probability of each possible move. How many are the possible moves? If the plaquette is weight $4$ these are $4 p$, if weight $3$ they are $3 p$. We can then represent a "move" by an edge, i.e., its label, and a power between $0, 1, \dots, p-1$ which represents the power of the corresponding odd stabilizer. So let us write a function that given a head label, gives back its edges. 